<a href="https://colab.research.google.com/github/shivtyam/Shivansh-2026B15410303/blob/main/Mobile_App_Usage_Analysis_Shivansh_Satyam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 24: Mobile App Usage & User Engagement Analysis

**Name:** Shivansh Satyam  
**Admission No.:** 2026b15410303

Place `24_mobile_app_usage.csv` in the same folder as this notebook, then run all cells. Edit the column map in the second code cell if your CSV uses different headers.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# ------------------------------------------------------------------
# COLUMN MAP - change the right-hand side to match your CSV headers
# ------------------------------------------------------------------
FILE = "24_mobile_app_usage.csv"
COL = {
    "session": "Session_ID",
    "user": "User_ID",
    "date": "Session_Date",
    "duration": "Session_Duration_Mins",
    "feature": "Feature_Used",
    "device": "Device_Type",
    "age_group": "Age_Group",   # if absent, script bins a numeric "Age" column
    "rating": "User_Rating",
}

In [ ]:
# ============ 1. LOAD AND INSPECT ============
df = pd.read_csv(FILE)
print("=== 1. INSPECTION ===")
print("Shape:", df.shape)
print(df.head(), "\n")
df.info()
print(df.describe(include="all").T, "\n")

missing_cols = [c for k, c in COL.items() if c not in df.columns and k != "age_group"]
if missing_cols:
    raise SystemExit(f"Column(s) not found: {missing_cols}. Update the COL map above.\n"
                     f"Available columns: {list(df.columns)}")

# Age group: use existing column, else bin numeric Age
if COL["age_group"] not in df.columns:
    if "Age" in df.columns:
        df[COL["age_group"]] = pd.cut(df["Age"], bins=[0, 17, 24, 34, 44, 54, 120],
                                      labels=["<18", "18-24", "25-34", "35-44", "45-54", "55+"])
    else:
        print("WARNING: no age-group or Age column; task 8 will be skipped.")

In [ ]:
# ============ 2. MISSING VALUES & DUPLICATES ============
print("\n=== 2. MISSING VALUES & DUPLICATES ===")
print(df.isna().sum())
dup_ids = df.duplicated(subset=COL["session"]).sum()
print(f"Duplicate {COL['session']} records: {dup_ids}")
df = df.drop_duplicates(subset=COL["session"], keep="first").copy()

In [ ]:
# ============ 3. HANDLE MISSING DURATION / RATING ============
# Justification:
# - Duration is right-skewed (a few very long sessions), so the MEDIAN is a safer fill
#   than the mean. We use the median within the same feature (behaviour differs by
#   feature), falling back to the overall median.
# - Non-positive durations are impossible, so they are treated as missing too.
# - Ratings are ordinal (e.g. 1-5), so the mean would produce non-valid values. We use
#   the per-feature median. Flag columns keep a record of what was imputed.
print("\n=== 3. MISSING VALUE TREATMENT ===")
dur, rat, feat = COL["duration"], COL["rating"], COL["feature"]
df.loc[df[dur] <= 0, dur] = np.nan
df["duration_imputed"] = df[dur].isna()
df["rating_imputed"] = df[rat].isna()
df[dur] = df[dur].fillna(df.groupby(feat)[dur].transform("median")).fillna(df[dur].median())
df[rat] = df[rat].fillna(df.groupby(feat)[rat].transform("median")).fillna(df[rat].median())
print("Imputed durations:", int(df["duration_imputed"].sum()),
      "| Imputed ratings:", int(df["rating_imputed"].sum()))
print("Remaining nulls in key columns:", int(df[[dur, rat]].isna().sum().sum()))

In [ ]:
# ============ 4. DATETIME FEATURES ============
print("\n=== 4. DATETIME ===")
df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")
bad_dates = df[COL["date"]].isna().sum()
if bad_dates:
    print(f"Dropping {bad_dates} rows with unparseable dates")
    df = df.dropna(subset=[COL["date"]])
df["Month"] = df[COL["date"]].dt.month_name()
df["Weekday"] = df[COL["date"]].dt.day_name()
df["Hour"] = df[COL["date"]].dt.hour
if df["Hour"].nunique() == 1:
    print("NOTE: Session_Date has no time component, so hour analysis (task 10) is not meaningful.")
print(df[[COL["date"], "Month", "Weekday", "Hour"]].head())

In [ ]:
# ============ 5. HEADLINE KPIs ============
print("\n=== 5. KPIs ===")
total_sessions = len(df)
unique_users = df[COL["user"]].nunique()
avg_duration = df[dur].mean()
print(f"Total sessions: {total_sessions}")
print(f"Unique users: {unique_users}")
print(f"Avg duration: {avg_duration:.2f} mins")
print(f"Sessions per user: {total_sessions / unique_users:.2f}")

In [ ]:
# ============ 6. TOP 10 FEATURES ============
print("\n=== 6. TOP 10 FEATURES ===")
top_features = df[feat].value_counts().head(10)
print(top_features)

In [ ]:
# ============ 7. DURATION BY DEVICE ============
print("\n=== 7. AVG DURATION BY DEVICE ===")
by_device = df.groupby(COL["device"])[dur].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
print(by_device.round(2))

In [ ]:
# ============ 8. AGE GROUP RANKING ============
print("\n=== 8. AGE GROUPS BY SESSIONS ===")
if COL["age_group"] in df.columns:
    age_rank = df[COL["age_group"]].value_counts().rename_axis("Age_Group").reset_index(name="Sessions")
    age_rank["Rank"] = age_rank["Sessions"].rank(ascending=False, method="min").astype(int)
    print(age_rank)

In [ ]:
# ============ 9. RATING BY FEATURE ============
print("\n=== 9. AVG RATING BY FEATURE ===")
rating_by_feat = df.groupby(feat)[rat].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(rating_by_feat.round(2))

In [ ]:
# ============ 10. PEAK HOURS & BUSIEST WEEKDAYS ============
print("\n=== 10. PEAK HOURS & WEEKDAYS ===")
hour_counts = df["Hour"].value_counts().sort_index()
print("Top 5 hours:\n", hour_counts.sort_values(ascending=False).head(5))
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = df["Weekday"].value_counts().reindex(order)
print("\nSessions by weekday:\n", weekday_counts)
print("Busiest weekday:", weekday_counts.idxmax())

In [ ]:
# ============ 11. LIST AND TUPLE ============
print("\n=== 11. LIST & TUPLE ===")
most_used_features = top_features.index.tolist()   # list: mutable, ordered by usage
DEVICE_CATEGORIES = tuple(sorted(df[COL["device"]].dropna().unique()))  # tuple: fixed set
print("Most-used features (list):", most_used_features)
print("Device categories (tuple):", DEVICE_CATEGORIES)

In [ ]:
# ============ 12. SET + FUNCTION ============
print("\n=== 12. SET & FUNCTION ===")
unique_user_set = set(df[COL["user"]])
print("Unique users via set:", len(unique_user_set))

def avg_duration_for_feature(data, feature_name):
    """Average session duration (mins) for one feature; None if the feature is not found."""
    subset = data.loc[data[feat] == feature_name, dur]
    return round(subset.mean(), 2) if not subset.empty else None

example = most_used_features[0]
print(f"Avg duration for '{example}':", avg_duration_for_feature(df, example))

In [ ]:
# ============ 13. NUMPY STATS ============
print("\n=== 13. NUMPY STATS ===")
d = df[dur].to_numpy()
print(f"Mean: {np.mean(d):.2f}\nMedian: {np.median(d):.2f}\nStd: {np.std(d, ddof=1):.2f}")

In [ ]:
# ============ 14. IQR OUTLIERS & ENGAGEMENT LABELS ============
print("\n=== 14. OUTLIERS & ENGAGEMENT ===")
q1, q3 = np.percentile(d, [25, 75])
iqr = q3 - q1
low_fence, high_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["Long_Session_Outlier"] = df[dur] > high_fence
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, upper fence={high_fence:.2f}")
print("Unusually long sessions:", int(df["Long_Session_Outlier"].sum()),
      f"({df['Long_Session_Outlier'].mean():.1%})")

# Rule (documented): based on session-duration quartiles
#   Low    : duration < Q1
#   Medium : Q1 <= duration < Q3
#   High   : duration >= Q3
def engagement_label(minutes):
    if minutes < q1:
        return "Low"
    if minutes < q3:
        return "Medium"
    return "High"

df["Engagement"] = df[dur].apply(engagement_label)
print(df["Engagement"].value_counts())

In [ ]:
# ============ 15. AppUser CLASS ============
print("\n=== 15. AppUser CLASS ===")
class AppUser:
    def __init__(self, user_id, device_type):
        self.user_id = user_id
        self.device_type = device_type
        self.sessions = []          # list of durations

    def add_session(self, minutes):
        self.sessions.append(minutes)

    def avg_session(self):
        return round(sum(self.sessions) / len(self.sessions), 2) if self.sessions else 0.0

    def __repr__(self):
        return f"AppUser(id={self.user_id}, device={self.device_type}, sessions={len(self.sessions)})"

sample_id = df[COL["user"]].iloc[0]
u = AppUser(sample_id, df.loc[df[COL["user"]] == sample_id, COL["device"]].iloc[0])
for m in df.loc[df[COL["user"]] == sample_id, dur]:
    u.add_session(m)
print(u, "| avg session:", u.avg_session(), "mins")

In [ ]:
# ============ RECOMMENDATIONS (data-driven; edit wording as needed) ============
print("\n=== RECOMMENDATIONS ===")
best_rated = rating_by_feat["mean"].idxmax()
worst_rated = rating_by_feat["mean"].idxmin()
long_device = by_device["mean"].idxmax()
short_device = by_device["mean"].idxmin()
peak_hour = int(hour_counts.idxmax())
busiest_day = weekday_counts.idxmax()
recs = [
    f"Promote '{best_rated}' (highest avg rating {rating_by_feat['mean'].max():.2f}) in onboarding and the home screen.",
    f"Investigate '{worst_rated}' (lowest avg rating {rating_by_feat['mean'].min():.2f}) through user feedback and usability testing.",
    f"Sessions on {short_device} average {by_device['mean'].min():.1f} mins versus {by_device['mean'].max():.1f} on {long_device}; "
    f"optimise the {short_device} experience (load time, layout, navigation).",
    f"Schedule notifications, content drops and releases around peak usage ({peak_hour}:00 and {busiest_day}s); "
    f"run maintenance outside those windows.",
    f"Re-engage Low-engagement users ({(df['Engagement'] == 'Low').mean():.0%} of sessions) with personalised prompts to try the "
    f"top features ({', '.join(most_used_features[:3])}), and study the {int(df['Long_Session_Outlier'].sum())} unusually long "
    f"sessions to see if they signal power users or usability problems.",
]
for i, r in enumerate(recs, 1):
    print(f"{i}. {r}")

In [ ]:
# ============ EXPORT DELIVERABLES ============
import os
os.makedirs("outputs", exist_ok=True)
top_features.rename_axis("Feature").reset_index(name="Sessions").assign(
    Share_Pct=lambda t: (t["Sessions"] / total_sessions * 100).round(2)
).to_csv("outputs/feature_popularity.csv", index=False)
by_device.round(2).to_csv("outputs/device_comparison.csv")
rating_by_feat.round(2).to_csv("outputs/rating_by_feature.csv")
if COL["age_group"] in df.columns:
    age_rank.to_csv("outputs/age_group_ranking.csv", index=False)
peak = pd.concat([
    hour_counts.rename_axis("Value").reset_index(name="Sessions").assign(Type="Hour"),
    weekday_counts.rename_axis("Value").reset_index(name="Sessions").assign(Type="Weekday"),
])[["Type", "Value", "Sessions"]]
peak.to_csv("outputs/peak_usage_summary.csv", index=False)
df[[COL["session"], COL["user"], dur, "Engagement", "Long_Session_Outlier"]].to_csv(
    "outputs/engagement_labels.csv", index=False)
pd.DataFrame({
    "Metric": ["Total sessions", "Unique users", "Avg duration (mins)", "Mean (NumPy)", "Median (NumPy)",
               "Std dev (NumPy)", "Q1", "Q3", "IQR", "Upper fence", "Outlier sessions"],
    "Value": [total_sessions, unique_users, round(avg_duration, 2), round(np.mean(d), 2),
              round(np.median(d), 2), round(np.std(d, ddof=1), 2), round(q1, 2), round(q3, 2),
              round(iqr, 2), round(high_fence, 2), int(df["Long_Session_Outlier"].sum())],
}).to_csv("outputs/session_statistics.csv", index=False)
with open("outputs/recommendations.txt", "w") as f:
    f.write("\n".join(f"{i}. {r}" for i, r in enumerate(recs, 1)))
print("Saved deliverable tables to the outputs/ folder")

df.to_csv("24_mobile_app_usage_cleaned.csv", index=False)
print("\nSaved cleaned data to 24_mobile_app_usage_cleaned.csv")